# FoundationStereo on Middlebury Eval3

This notebook reproduces the FoundationStereo Middlebury experiment without cloning the full GitHub repository. Upload this notebook together with `foundationstereo_source_bundle.zip`.

Protocol: half-resolution Middlebury, `scale=1.0`, 32 refinement iterations, mixed precision, all 30 input pairs, and non-occluded metrics on the 15 public training pairs.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown', 'omegaconf', 'imgaug', 'ruamel.yaml', 'trimesh', 'imageio', 'open3d', 'transformations', 'einops'], check=True)

try:
    import torch
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
except Exception as exc:
    print(exc)
    
BUNDLE = Path('/content/foundationstereo_bundle')
SOURCE_ZIP = Path('/content/foundationstereo_source_bundle.zip')
if not SOURCE_ZIP.exists():
    from google.colab import files
    print('Upload foundationstereo_source_bundle.zip')
    uploaded = files.upload()
    if 'foundationstereo_source_bundle.zip' not in uploaded:
        raise FileNotFoundError('Upload foundationstereo_source_bundle.zip')
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir(parents=True)
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    archive.extractall(BUNDLE)
print('Source files:', sorted(p.name for p in BUNDLE.iterdir()))

In [ ]:
import requests

ROOT = Path('/content/foundationstereo_middlebury')
DATA = ROOT / 'data' / 'middlebury'
CHECKPOINT = ROOT / 'checkpoints' / '23-51-11'
OUTPUT = ROOT / 'outputs' / 'middlebury_half'
DATA.mkdir(parents=True, exist_ok=True)
CHECKPOINT.mkdir(parents=True, exist_ok=True)

def download(url, path):
    if path.exists() and path.stat().st_size > 0:
        print('Already downloaded', path.name)
        return
    print('Downloading', path.name)
    headers = {'User-Agent': 'Mozilla/5.0 (Colab; research reproduction)'}
    response = requests.get(url, headers=headers, stream=True, timeout=60)
    try:
        response.raise_for_status()
        with open(path, 'wb') as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
    except requests.HTTPError as error:
        if path.exists():
            path.unlink()
        print(f'Requests failed ({error}); retrying with wget...')
        subprocess.run(['wget', '--no-verbose', '--user-agent=Mozilla/5.0', '-O', str(path), url], check=True)
    finally:
        response.close()

download('https://vision.middlebury.edu/stereo/submit3/zip/MiddEval3-data-H.zip', DATA / 'MiddEval3-data-H.zip')
download('https://vision.middlebury.edu/stereo/submit3/zip/MiddEval3-GT0-H.zip', DATA / 'MiddEval3-GT0-H.zip')
subprocess.run([sys.executable, '-m', 'gdown', 'https://drive.google.com/uc?id=1tidGICH1_kTUUqi42aboKscuMY4IK_Xr', '-O', str(CHECKPOINT / 'cfg.yaml')], check=True)
subprocess.run([sys.executable, '-m', 'gdown', 'https://drive.google.com/uc?id=1Yh_2o9QCUrVqZrnAXZ7RUr0zTp3JrMKe', '-O', str(CHECKPOINT / 'model_best_bp2.pth')], check=True)

for archive_name, folder in [('MiddEval3-data-H.zip', 'data-H'), ('MiddEval3-GT0-H.zip', 'gt0-H')]:
    archive_path = DATA / archive_name
    extract_path = DATA / folder
    if not (extract_path / 'MiddEval3').exists():
        extract_path.mkdir(exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(extract_path)
print('Prepared data and checkpoint.')

In [ ]:
import csv, time
import cv2, imageio.v2 as imageio, numpy as np, torch
from omegaconf import OmegaConf

source_roots = [BUNDLE] + [path.parent for path in BUNDLE.rglob('core/foundation_stereo.py')]
SOURCE_ROOT = next((path for path in source_roots if (path / 'core' / 'foundation_stereo.py').exists()), None)
if SOURCE_ROOT is None:
    raise FileNotFoundError(f'Could not find core/foundation_stereo.py under {BUNDLE}. Contents: {list(BUNDLE.rglob("*") )[:20]}')
sys.path.insert(0, str(SOURCE_ROOT))
print('Using source root:', SOURCE_ROOT)
from core.foundation_stereo import FoundationStereo
from core.utils.utils import InputPadder

cfg = OmegaConf.load(CHECKPOINT / 'cfg.yaml')
if 'vit_size' not in cfg:
    cfg['vit_size'] = 'vitl'
cfg['mixed_precision'] = True
model = FoundationStereo(cfg)
state = torch.load(CHECKPOINT / 'model_best_bp2.pth', map_location='cpu', weights_only=False)
model.load_state_dict(state['model'])
model.cuda().eval()

def write_pfm(path, image):
    image = np.asarray(image, dtype=np.float32)
    with open(path, 'wb') as f:
        f.write(b'Pf\n')
        f.write(f'{image.shape[1]} {image.shape[0]}\n'.encode())
        f.write(b'-1.0\n')
        np.flipud(image).tofile(f)

def run_pair(left_path, right_path):
    left, right = imageio.imread(left_path), imageio.imread(right_path)
    h, w = left.shape[:2]
    x0 = torch.as_tensor(left).cuda().float()[None].permute(0, 3, 1, 2)
    x1 = torch.as_tensor(right).cuda().float()[None].permute(0, 3, 1, 2)
    padder = InputPadder(x0.shape, divis_by=32, force_square=False)
    x0, x1 = padder.pad(x0, x1)
    start = time.perf_counter()
    with torch.no_grad(), torch.cuda.amp.autocast(True):
        disp = model.forward(x0, x1, iters=32, test_mode=True)
    elapsed = time.perf_counter() - start
    disp = padder.unpad(disp.float()).detach().cpu().numpy().reshape(h, w).astype(np.float32)
    return disp, elapsed

data_root = DATA / 'data-H' / 'MiddEval3'
pred_root = OUTPUT / 'predictions'
pred_root.mkdir(parents=True, exist_ok=True)
rows = []
for split in ['trainingH', 'testH']:
    for scene_dir in sorted((data_root / split).iterdir()):
        if not scene_dir.is_dir():
            continue
        out_dir = pred_root / split / scene_dir.name
        out_dir.mkdir(parents=True, exist_ok=True)
        pred_path = out_dir / 'disp0FS-paper.pfm'
        time_path = out_dir / 'timeFS-paper.txt'
        if pred_path.exists() and time_path.exists():
            print('[skip]', split, scene_dir.name)
            continue
        disp, elapsed = run_pair(scene_dir / 'im0.png', scene_dir / 'im1.png')
        write_pfm(pred_path, disp)
        time_path.write_text(f'{elapsed:.6f}\n')
        rows.append({'split': split, 'scene': scene_dir.name, 'seconds': elapsed})
        print('[done]', split, scene_dir.name, f'{elapsed:.2f}s')
with open(pred_root / 'inference_summary.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['split', 'scene', 'seconds'])
    writer.writeheader(); writer.writerows(rows)
print('Inference complete.')

In [ ]:
def read_pfm(path):
    with open(path, 'rb') as f:
        if f.readline().strip() != b'Pf':
            raise ValueError(f'Expected grayscale PFM: {path}')
        width, height = map(int, f.readline().split())
        scale = float(f.readline())
        endian = '<' if scale < 0 else '>'
        values = np.fromfile(f, endian + 'f')
    return np.flipud(values.reshape(height, width)).astype(np.float32)

gt_root = DATA / 'gt0-H' / 'MiddEval3'
metric_rows = []
for scene_dir in sorted((gt_root / 'trainingH').iterdir()):
    if not scene_dir.is_dir():
        continue
    pred = read_pfm(pred_root / 'trainingH' / scene_dir.name / 'disp0FS-paper.pfm')
    gt = read_pfm(scene_dir / 'disp0GT.pfm')
    mask = imageio.imread(scene_dir / 'mask0nocc.png') == 255
    valid = np.isfinite(pred) & np.isfinite(gt) & mask
    error = np.abs(pred[valid] - gt[valid])
    metric_rows.append({'scene': scene_dir.name, 'valid_pixels': int(valid.sum()), 'epe': float(error.mean()), 'rmse': float(np.sqrt(np.mean(error ** 2))), 'bp0.5': float(np.mean(error > 0.5) * 100), 'bp1.0': float(np.mean(error > 1.0) * 100), 'bp2.0': float(np.mean(error > 2.0) * 100), 'bp4.0': float(np.mean(error > 4.0) * 100)})
metrics_dir = OUTPUT / 'metrics'; metrics_dir.mkdir(parents=True, exist_ok=True)
metrics_path = metrics_dir / 'middlebury_training_half_nonocc.csv'
with open(metrics_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(metric_rows[0])); writer.writeheader(); writer.writerows(metric_rows)
print('Paper-style local result: half-resolution, non-occluded, 15 training scenes')
for key in ['epe', 'rmse', 'bp0.5', 'bp1.0', 'bp2.0', 'bp4.0']:
    print(f'{key}: {np.mean([row[key] for row in metric_rows]):.4f}')
print('Metrics saved to', metrics_path)

In [ ]:
archive_path = Path('/content/foundationstereo_middlebury_results.zip')
if archive_path.exists(): archive_path.unlink()
shutil.make_archive(str(archive_path.with_suffix('')), 'zip', OUTPUT)
print(archive_path)
from google.colab import files
files.download(str(archive_path))